<a href="https://colab.research.google.com/github/islam37/S-SLTE-Thesis/blob/main/notebooks/08_boilerplate_contamination_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# S-SLTE | Boilerplate Contamination Audit
# Stage 8 — Dataset inspection
# Read-only: no reference labels, no dataset modifications.

from pathlib import Path
from google.colab import drive

import pandas as pd
import numpy as np
import hashlib

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

def load_text_only(relative_path):
    path = BASE / relative_path
    assert path.exists(), f"Missing file: {path}"

    df = pd.read_csv(
        path,
        usecols=["record_id", "text"]
    )

    assert df["record_id"].is_unique
    assert df["text"].notna().all()
    return df

anchor = load_text_only("data/splits/anchor.csv")
development = load_text_only("data/splits/development.csv")
noisy = load_text_only("data/splits/noisy_training.csv")
pilot = load_text_only("data/pilot/pilot_500.csv")

assert (
    len(anchor),
    len(development),
    len(noisy),
    len(pilot)
) == (2000, 1800, 16200, 500)

# Reproduce the existing proposed holdout.
noisy_ids = set(noisy["record_id"].astype(int))
pilot_ids = set(pilot["record_id"].astype(int))

assert pilot_ids.issubset(noisy_ids)

available_ids = np.array(
    sorted(noisy_ids - pilot_ids),
    dtype=np.int64
)

rng = np.random.default_rng(20261025)

holdout_ids = set(
    rng.choice(
        available_ids,
        size=2000,
        replace=False
    ).tolist()
)

def fingerprint_ids(ids):
    payload = "\n".join(
        str(int(x)) for x in sorted(ids)
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

EXPECTED_HOLDOUT_SHA = (
    "37bd15b5fb600e7cfc392c936fcb"
    "187b0a39313c3502098c0e6a403153057051"
)

assert fingerprint_ids(holdout_ids) == EXPECTED_HOLDOUT_SHA

holdout = noisy[
    noisy["record_id"].isin(holdout_ids)
].copy()

training = noisy[
    ~noisy["record_id"].isin(
        pilot_ids | holdout_ids
    )
].copy()

assert len(holdout) == 2000
assert len(training) == 13700

# Audit raw text without transforming it.
partitions = {
    "anchor": anchor,
    "development": development,
    "pilot": pilot,
    "proposed_holdout": holdout,
    "training_candidate": training
}

audit_rows = []

for name, df in partitions.items():
    texts = df["text"].astype(str)

    advertisement = texts.str.contains(
        r"ADVERTISEMENT|ad\.doubleclick\.net",
        case=False,
        regex=True
    )

    html_markup = texts.str.contains(
        r"<(?:/?[a-z][^>]*|!--.*?)>|"
        r"&lt;/?[a-z]|&gt;|&amp;",
        case=False,
        regex=True
    )

    doubleclick = texts.str.contains(
        r"ad\.doubleclick\.net",
        case=False,
        regex=True
    )

    audit_rows.append({
        "partition": name,
        "records": len(df),
        "ad_flagged": int(advertisement.sum()),
        "html_flagged": int(html_markup.sum()),
        "doubleclick_flagged": int(doubleclick.sum()),
        "ad_rate_percent": round(
            advertisement.mean() * 100, 2
        )
    })

audit_summary = pd.DataFrame(audit_rows)

print("BOILERPLATE CONTAMINATION BASELINE AUDIT")
print("----------------------------------------")
print("Holdout fingerprint: VERIFIED")
print("Reference labels accessed: NO")

display(audit_summary)

print("\nVALIDATION STATUS")
print("Total proposed holdout records:", len(holdout))
print("Total pre-dedup training records:", len(training))
print("Text preprocessing applied: NO")
print("Dataset files modified: NO")
print("Holdout frozen: NO")
print("Audit status: BASELINE_COMPLETED")


Mounted at /content/drive
BOILERPLATE CONTAMINATION BASELINE AUDIT
----------------------------------------
Holdout fingerprint: VERIFIED
Reference labels accessed: NO


,partition,records,ad_flagged,html_flagged,doubleclick_flagged,ad_rate_percent
0,anchor,2000,1,75,0,0.05
1,development,1800,3,78,2,0.17
2,pilot,500,1,25,0,0.20
3,proposed_holdout,2000,5,109,5,0.25
4,training_candidate,13700,8,618,5,0.06



VALIDATION STATUS
Total proposed holdout records: 2000
Total pre-dedup training records: 13700
Text preprocessing applied: NO
Dataset files modified: NO
Holdout frozen: NO
Audit status: BASELINE_COMPLETED


In [2]:

# S-SLTE | Advertisement Structure Inspection
# Read-only diagnostic: no article deletion or modification.

import re
import html
import hashlib
import pandas as pd

AD_PATTERN = re.compile(
    r"<p>\s*ADVERTISEMENT\s*</p>"
    r"|ADVERTISEMENT"
    r"|ad\.doubleclick\.net",
    flags=re.IGNORECASE
)

records = []

for partition_name, df in partitions.items():

    for row in df.itertuples(index=False):

        original = str(row.text)
        decoded = html.unescape(original)
        match = AD_PATTERN.search(decoded)

        if match is None:
            continue

        prefix = decoded[:match.start()]
        suffix = decoded[match.start():]

        normalized_suffix = re.sub(
            r"\s+", " ", suffix
        ).strip().casefold()

        signature = hashlib.sha256(
            normalized_suffix.encode("utf-8")
        ).hexdigest()[:16]

        records.append({
            "partition": partition_name,
            "record_id": int(row.record_id),
            "total_chars": len(decoded),
            "article_chars_before_ad": len(prefix),
            "chars_from_ad_onward": len(suffix),
            "ad_start_percent": round(
                100 * match.start() / max(len(decoded), 1),
                2
            ),
            "has_doubleclick": bool(
                re.search(
                    r"ad\.doubleclick\.net",
                    decoded,
                    flags=re.IGNORECASE
                )
            ),
            "ad_tail_signature": signature
        })

ad_structure = pd.DataFrame(records)

assert len(ad_structure) == int(
    audit_summary["ad_flagged"].sum()
)

assert ad_structure["record_id"].is_unique

print("ADVERTISEMENT STRUCTURE INSPECTION")
print("----------------------------------")
print("Flagged articles:", len(ad_structure))
print(
    "Articles containing DoubleClick:",
    int(ad_structure["has_doubleclick"].sum())
)

print("\nFLAGGED RECORDS BY PARTITION")
display(
    ad_structure.groupby("partition")
    .agg(
        records=("record_id", "count"),
        median_ad_start_pct=(
            "ad_start_percent", "median"
        ),
        min_article_chars_before_ad=(
            "article_chars_before_ad", "min"
        )
    )
    .round(2)
    .reset_index()
)

print("\nREPEATED ADVERTISEMENT TAIL SIGNATURES")

repeated_signatures = (
    ad_structure.groupby("ad_tail_signature")
    .agg(
        records=("record_id", "count"),
        partitions=(
            "partition",
            lambda x: ", ".join(sorted(set(x)))
        )
    )
    .query("records > 1")
    .sort_values(
        "records", ascending=False
    )
    .reset_index()
)

display(repeated_signatures)

print("\nFLAGGED HOLDOUT RECORDS")
display(
    ad_structure.loc[
        ad_structure["partition"] == "proposed_holdout",
        [
            "record_id",
            "ad_start_percent",
            "article_chars_before_ad",
            "chars_from_ad_onward",
            "ad_tail_signature"
        ]
    ].reset_index(drop=True)
)

print("\nVALIDATION STATUS")
print("Original article texts displayed: NO")
print("Article cleaning applied: NO")
print("Original dataset files modified: NO")
print("Reference labels accessed: NO")
print("Holdout frozen: NO")
print("Status: STRUCTURE_INSPECTED")


ADVERTISEMENT STRUCTURE INSPECTION
----------------------------------
Flagged articles: 18
Articles containing DoubleClick: 12

FLAGGED RECORDS BY PARTITION


,partition,records,median_ad_start_pct,min_article_chars_before_ad
0,anchor,1,91.84,270
1,development,3,28.26,168
2,pilot,1,63.53,162
3,proposed_holdout,5,20.82,156
4,training_candidate,8,47.25,106



REPEATED ADVERTISEMENT TAIL SIGNATURES


,ad_tail_signature,records,partitions
0,f2b1f9f363c8b8f7,6,"development, proposed_holdout, training_candidate"
1,7a6f3a43563b4efb,2,"development, training_candidate"
2,bdc3818233cb2360,2,"proposed_holdout, training_candidate"



FLAGGED HOLDOUT RECORDS


,record_id,ad_start_percent,article_chars_before_ad,chars_from_ad_onward,ad_tail_signature
0,65851,43.40,342,446,bdc3818233cb2360
1,95309,17.45,156,738,f2b1f9f363c8b8f7
2,106486,23.20,223,738,f2b1f9f363c8b8f7
3,111618,19.26,176,738,f2b1f9f363c8b8f7
4,112808,20.82,194,738,f2b1f9f363c8b8f7



VALIDATION STATUS
Original article texts displayed: NO
Article cleaning applied: NO
Original dataset files modified: NO
Reference labels accessed: NO
Holdout frozen: NO
Status: STRUCTURE_INSPECTED


In [3]:

# S-SLTE | Controlled Advertisement Removal Test
# In-memory screening candidates only.
# No reference labels, original file edits, or exports.

import re
import html
import itertools
import pandas as pd
from difflib import SequenceMatcher

AD_BLOCK = re.compile(
    r"<p\b[^>]*>\s*ADVERTISEMENT\s*</p\s*>",
    flags=re.IGNORECASE
)

def build_screening_candidate(raw_text):
    decoded = html.unescape(str(raw_text))
    match = AD_BLOCK.search(decoded)

    if match is None:
        return decoded, "NO_STRUCTURED_MARKER"

    prefix = decoded[:match.start()].strip()

    if len(prefix) < 100:
        return decoded, "SHORT_PREFIX_REVIEW"

    # Proposed screening candidate only.
    # Tail removal may discard legitimate content.
    return prefix, "TAIL_REMOVAL_CANDIDATE"


# Inspect only the previously flagged 18 records.
all_partition_texts = pd.concat(
    [
        df.assign(partition=name)
        for name, df in partitions.items()
    ],
    ignore_index=True
)

assert all_partition_texts["record_id"].is_unique

flagged_ids = set(ad_structure["record_id"].astype(int))

flagged_texts = all_partition_texts[
    all_partition_texts["record_id"].isin(flagged_ids)
].copy()

assert len(flagged_texts) == 18

candidate_lookup = {}
results = []

for row in flagged_texts.itertuples(index=False):

    decoded = html.unescape(str(row.text))

    candidate, status = build_screening_candidate(
        row.text
    )

    record_id = int(row.record_id)

    candidate_lookup[record_id] = candidate

    results.append({
        "record_id": record_id,
        "partition": row.partition,
        "status": status,
        "raw_chars": len(decoded),
        "candidate_chars": len(candidate),
        "removed_chars": len(decoded) - len(candidate),
        "retained_percent": round(
            100 * len(candidate) / max(len(decoded), 1),
            2
        )
    })

cleaning_audit = pd.DataFrame(results)

print("CONTROLLED ADVERTISEMENT REMOVAL TEST")
print("-------------------------------------")

print("\nSCREENING CANDIDATE STATUS")
display(
    cleaning_audit["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="records")
)

print("\nFLAGGED HOLDOUT RECORDS")
display(
    cleaning_audit[
        cleaning_audit["partition"] == "proposed_holdout"
    ][
        [
            "record_id",
            "status",
            "raw_chars",
            "candidate_chars",
            "retained_percent"
        ]
    ].sort_values("record_id").reset_index(drop=True)
)


# Compare the suspicious four-record cluster.
cluster_ids = [95309, 106486, 111618, 112808]

text_lookup = all_partition_texts.set_index(
    "record_id"
)["text"]

assert set(cluster_ids).issubset(text_lookup.index)

def normalize_for_comparison(text):
    return re.sub(
        r"\s+", " ",
        str(text).casefold()
    ).strip()

def similarity(a, b):
    a = normalize_for_comparison(a)
    b = normalize_for_comparison(b)

    return SequenceMatcher(
        None, a, b, autojunk=False
    ).ratio()

pair_results = []

for record_a, record_b in itertools.combinations(
    cluster_ids, 2
):
    raw_a = html.unescape(text_lookup.loc[record_a])
    raw_b = html.unescape(text_lookup.loc[record_b])

    candidate_a = candidate_lookup[record_a]
    candidate_b = candidate_lookup[record_b]

    pair_results.append({
        "record_a": record_a,
        "record_b": record_b,
        "raw_sequence_similarity": round(
            similarity(raw_a, raw_b), 4
        ),
        "candidate_sequence_similarity": round(
            similarity(candidate_a, candidate_b), 4
        )
    })

cluster_comparison = pd.DataFrame(pair_results)

print("\nSUSPICIOUS CLUSTER: RAW VS CANDIDATE")
display(cluster_comparison)

print("\nVALIDATION STATUS")
print("Original articles displayed: NO")
print("Original datasets modified: NO")
print("Candidate screening texts saved: NO")
print("Final cleaning policy approved: NO")
print("Reference labels accessed: NO")
print("Holdout frozen: NO")
print("Status: PREPROCESSING_CANDIDATE_TESTED")


CONTROLLED ADVERTISEMENT REMOVAL TEST
-------------------------------------

SCREENING CANDIDATE STATUS


,status,records
0,TAIL_REMOVAL_CANDIDATE,12
1,NO_STRUCTURED_MARKER,6



FLAGGED HOLDOUT RECORDS


,record_id,status,raw_chars,candidate_chars,retained_percent
0,65851,TAIL_REMOVAL_CANDIDATE,788,342,43.40
1,95309,TAIL_REMOVAL_CANDIDATE,894,156,17.45
2,106486,TAIL_REMOVAL_CANDIDATE,961,223,23.20
3,111618,TAIL_REMOVAL_CANDIDATE,914,176,19.26
4,112808,TAIL_REMOVAL_CANDIDATE,932,194,20.82



SUSPICIOUS CLUSTER: RAW VS CANDIDATE


,record_a,record_b,raw_sequence_similarity,candidate_sequence_similarity
0,95309,106486,0.8367,0.2005
1,95309,111618,0.8662,0.2711
2,95309,112808,0.8346,0.1371
3,106486,111618,0.8256,0.1805
4,106486,112808,0.8357,0.2542
5,111618,112808,0.8342,0.1730



VALIDATION STATUS
Original articles displayed: NO
Original datasets modified: NO
Candidate screening texts saved: NO
Final cleaning policy approved: NO
Reference labels accessed: NO
Holdout frozen: NO
Status: PREPROCESSING_CANDIDATE_TESTED


In [4]:

# S-SLTE | Advertisement Removal Safety Audit

import re
import html
import pandas as pd

def compact_preview(value, limit=140):
    value = html.unescape(str(value))
    value = re.sub(r"<[^>]*>", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value[:limit]

safety_rows = []

for row in flagged_texts.itertuples(index=False):
    raw = html.unescape(str(row.text))

    structured = AD_BLOCK.search(raw)
    any_ad = AD_PATTERN.search(raw)

    assert any_ad is not None, (
        f"Missing ad marker for record {row.record_id}"
    )

    if structured:
        prefix = raw[:structured.start()]
        tail = raw[structured.start():]

        status = "STRUCTURED_AD_TAIL"
        preview_start = compact_preview(tail[:350])
        preview_end = compact_preview(tail[-250:])

    else:
        prefix = raw[:any_ad.start()]
        tail = raw[any_ad.start():]

        status = "NON_STRUCTURED_MARKER"
        preview_start = compact_preview(tail[:250])
        preview_end = compact_preview(tail[-200:])

    safety_rows.append({
        "record_id": int(row.record_id),
        "partition": row.partition,
        "marker_type": status,
        "prefix_chars": len(prefix),
        "tail_chars": len(tail),
        "tail_start_preview": preview_start,
        "tail_end_preview": preview_end
    })

safety_audit = pd.DataFrame(safety_rows)

assert len(safety_audit) == 18
assert safety_audit["record_id"].is_unique

assert (
    safety_audit["marker_type"]
    .value_counts()
    .get("STRUCTURED_AD_TAIL", 0)
) == 12

print("ADVERTISEMENT CONTENT PRESERVATION AUDIT")
print("---------------------------------------")

print("\nSTRUCTURED REMOVAL CANDIDATES")
display(
    safety_audit[
        safety_audit["marker_type"]
        == "STRUCTURED_AD_TAIL"
    ].sort_values("record_id").reset_index(drop=True)
)

print("\nUNHANDLED ADVERTISEMENT MARKERS")
display(
    safety_audit[
        safety_audit["marker_type"]
        == "NON_STRUCTURED_MARKER"
    ].sort_values("record_id").reset_index(drop=True)
)

print("\nVALIDATION STATUS")
print("Article removals: 0")
print("Original files modified: NO")
print("Reference labels accessed: NO")
print("Cleaning rule finalized: NO")
print("Holdout frozen: NO")
print("Manual tail-content review: REQUIRED")


ADVERTISEMENT CONTENT PRESERVATION AUDIT
---------------------------------------

STRUCTURED REMOVAL CANDIDATES


,record_id,partition,marker_type,prefix_chars,tail_chars,tail_start_preview,tail_end_preview
0,368,development,STRUCTURED_AD_TAIL,230,584,"ADVERTISEMENT <a href=""http://ad.doubleclick.n...",Series met=boardroom me=E P_Creative=P_InfoW_R...
1,9411,training_candidate,STRUCTURED_AD_TAIL,325,584,"ADVERTISEMENT <a href=""http://ad.doubleclick.n...",Series met=boardroom me=E P_Creative=P_InfoW_R...
2,49078,training_candidate,STRUCTURED_AD_TAIL,384,446,ADVERTISEMENT SBC Case Study: Crate Barrel Wha...,28975;9651165;a?http://www.infoworld.com/spotl...
3,65851,proposed_holdout,STRUCTURED_AD_TAIL,342,446,ADVERTISEMENT SBC Case Study: Crate Barrel Wha...,28975;9651165;a?http://www.infoworld.com/spotl...
4,82265,training_candidate,STRUCTURED_AD_TAIL,190,425,ADVERTISEMENT SBC Datacomm White Paper Find ou...,oubleclick.net/clk;11714255;9651165;g?http://i...
5,95309,proposed_holdout,STRUCTURED_AD_TAIL,156,738,ADVERTISEMENT Solaris 10(TM) OS: Position your...,X platform.\Like the fact you can run Linux an...
6,96745,training_candidate,STRUCTURED_AD_TAIL,462,496,ADVERTISEMENT White Paper: Converged Services ...,"ghts/sbc/main.html?lpid0101035400730402idlp"">W..."
7,105416,development,STRUCTURED_AD_TAIL,168,738,ADVERTISEMENT Solaris 10(TM) OS: Position your...,X platform.\Like the fact you can run Linux an...
8,106486,proposed_holdout,STRUCTURED_AD_TAIL,223,738,ADVERTISEMENT Solaris 10(TM) OS: Position your...,X platform.\Like the fact you can run Linux an...
9,111618,proposed_holdout,STRUCTURED_AD_TAIL,176,738,ADVERTISEMENT Solaris 10(TM) OS: Position your...,X platform.\Like the fact you can run Linux an...



UNHANDLED ADVERTISEMENT MARKERS


,record_id,partition,marker_type,prefix_chars,tail_chars,tail_start_preview,tail_end_preview
0,763,development,NON_STRUCTURED_MARKER,210,220,advertisements and speeches say about Greece's...,"peeches say about Greece's modern, Western ori..."
1,10504,anchor,NON_STRUCTURED_MARKER,270,24,advertisement forthwith.,advertisement forthwith.
2,28789,pilot,NON_STRUCTURED_MARKER,162,93,advertisements meant to energize two groups th...,advertisements meant to energize two groups th...
3,31145,training_candidate,NON_STRUCTURED_MARKER,106,113,advertisements about its electric rates. The p...,advertisements about its electric rates. The p...
4,63575,training_candidate,NON_STRUCTURED_MARKER,171,41,advertisement of the exercise phenomenon.,advertisement of the exercise phenomenon.
5,79094,training_candidate,NON_STRUCTURED_MARKER,175,57,advertisements via instant-messaging tools or ...,advertisements via instant-messaging tools or ...



VALIDATION STATUS
Article removals: 0
Original files modified: NO
Reference labels accessed: NO
Cleaning rule finalized: NO
Holdout frozen: NO
Manual tail-content review: REQUIRED


In [5]:

# S-SLTE | Controlled TF-IDF Re-screening

import numpy as np
import pandas as pd
import html
import itertools

from sklearn.feature_extraction.text import TfidfVectorizer

# Restore the previously verified candidate training view.
EXACT_EXCLUSIONS = {9266, 43231, 68708, 79605}

clean_training = training[
    ~training["record_id"].isin(EXACT_EXCLUSIONS)
].copy()

assert len(clean_training) == 13696

comparison_df = pd.concat(
    [anchor, development, clean_training, pilot],
    ignore_index=True
)

assert len(comparison_df) == 17996
assert comparison_df["record_id"].is_unique

holdout_screen = holdout.reset_index(drop=True)
comparison_screen = comparison_df.reset_index(drop=True)

all_screen = pd.concat(
    [holdout_screen, comparison_screen],
    ignore_index=True
)

assert len(all_screen) == 19996

def candidate_clean(text):
    raw = str(text)
    decoded = html.unescape(raw)
    match = AD_BLOCK.search(decoded)

    if match is None:
        return raw

    prefix = decoded[:match.start()].strip()

    if len(prefix) < 100:
        return raw

    return prefix

raw_texts = all_screen["text"].astype(str).tolist()

candidate_texts = [
    candidate_clean(text)
    for text in raw_texts
]

changed_ids = [
    int(all_screen.iloc[i]["record_id"])
    for i, (a, b) in enumerate(
        zip(raw_texts, candidate_texts)
    )
    if a != b
]

assert len(changed_ids) == 12
assert set(changed_ids).issubset(flagged_ids)

HOLDOUT_N = len(holdout_screen)
TOP_K = 5
THRESHOLD = 0.85
BATCH_SIZE = 100

holdout_record_ids = (
    holdout_screen["record_id"].astype(int).to_numpy()
)

comparison_record_ids = (
    comparison_screen["record_id"].astype(int).to_numpy()
)

def scan_pairs(X_query, X_target, query_ids,
               target_ids, internal=False):

    pairs = {}

    for start in range(0, len(query_ids), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(query_ids))

        sim = (
            X_query[start:end] @ X_target.T
        ).toarray()

        if internal:
            sim[
                np.arange(end - start),
                np.arange(start, end)
            ] = -1.0

        top_indices = np.argpartition(
            sim, -TOP_K, axis=1
        )[:, -TOP_K:]

        for local_i in range(end - start):
            a = int(query_ids[start + local_i])

            for j in top_indices[local_i]:
                score = float(sim[local_i, j])

                if score < THRESHOLD:
                    continue

                b = int(target_ids[j])

                key = (
                    tuple(sorted((a, b)))
                    if internal else (a, b)
                )

                pairs[key] = max(
                    pairs.get(key, -1.0),
                    score
                )

    return pairs

def run_screening(texts, label):
    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_features=80000,
        sublinear_tf=True,
        dtype=np.float32
    )

    matrix = vectorizer.fit_transform(texts)

    X_hold = matrix[:HOLDOUT_N]
    X_other = matrix[HOLDOUT_N:]

    external = scan_pairs(
        X_hold, X_other,
        holdout_record_ids,
        comparison_record_ids
    )

    internal = scan_pairs(
        X_hold, X_hold,
        holdout_record_ids,
        holdout_record_ids,
        internal=True
    )

    # Direct cosine values for the suspicious cluster.
    lookup = {
        int(record_id): idx
        for idx, record_id in enumerate(holdout_record_ids)
    }

    cluster_scores = {}

    cluster_ids = [95309, 106486, 111618, 112808]

    for a, b in itertools.combinations(cluster_ids, 2):
        score = (
            X_hold[lookup[a]] @ X_hold[lookup[b]].T
        ).toarray()[0, 0]

        cluster_scores[(a, b)] = float(score)

    print(f"{label} SCREENING COMPLETED")
    print("External candidate pairs:", len(external))
    print("Internal candidate pairs:", len(internal))

    return external, internal, cluster_scores


raw_external, raw_internal, raw_cluster = (
    run_screening(raw_texts, "RAW")
)

clean_external, clean_internal, clean_cluster = (
    run_screening(candidate_texts, "CANDIDATE-CLEANED")
)

screening_summary = pd.DataFrame([
    {
        "version": "RAW",
        "external_pairs": len(raw_external),
        "internal_pairs": len(raw_internal)
    },
    {
        "version": "CANDIDATE_CLEANED",
        "external_pairs": len(clean_external),
        "internal_pairs": len(clean_internal)
    }
])

cluster_results = pd.DataFrame([
    {
        "record_a": a,
        "record_b": b,
        "raw_cosine": round(raw_cluster[(a, b)], 4),
        "cleaned_cosine": round(clean_cluster[(a, b)], 4)
    }
    for a, b in itertools.combinations(
        [95309, 106486, 111618, 112808], 2
    )
])

print("\nSCREENING COMPARISON")
display(screening_summary)

print("\nSUSPICIOUS CLUSTER — COSINE COMPARISON")
display(cluster_results)

print("\nCANDIDATE SET CHANGES")
print(
    "External pairs no longer flagged:",
    len(set(raw_external) - set(clean_external))
)
print(
    "New external pairs flagged:",
    len(set(clean_external) - set(raw_external))
)
print(
    "Internal pairs no longer flagged:",
    len(set(raw_internal) - set(clean_internal))
)
print(
    "New internal pairs flagged:",
    len(set(clean_internal) - set(raw_internal))
)

print("\nVALIDATION STATUS")
print("Candidate text changes:", len(changed_ids))
print("Original datasets modified: NO")
print("Private reference labels accessed: NO")
print("Human decisions automatically changed: NO")
print("Holdout frozen: NO")
print("Screening policy: EXPERIMENTAL")


RAW SCREENING COMPLETED
External candidate pairs: 63
Internal candidate pairs: 8
CANDIDATE-CLEANED SCREENING COMPLETED
External candidate pairs: 54
Internal candidate pairs: 2

SCREENING COMPARISON


,version,external_pairs,internal_pairs
0,RAW,63,8
1,CANDIDATE_CLEANED,54,2



SUSPICIOUS CLUSTER — COSINE COMPARISON


,record_a,record_b,raw_cosine,cleaned_cosine
0,95309,106486,0.8680,0.0061
1,95309,111618,0.8767,0.0099
2,95309,112808,0.8921,0.0020
3,106486,111618,0.8611,0.0000
4,106486,112808,0.8778,0.0069
5,111618,112808,0.8862,0.0054



CANDIDATE SET CHANGES
External pairs no longer flagged: 9
New external pairs flagged: 0
Internal pairs no longer flagged: 6
New internal pairs flagged: 0

VALIDATION STATUS
Candidate text changes: 12
Original datasets modified: NO
Private reference labels accessed: NO
Human decisions automatically changed: NO
Holdout frozen: NO
Screening policy: EXPERIMENTAL


In [6]:

# S-SLTE | Candidate Pair Change Audit

import pandas as pd

removed_external = sorted(
    set(raw_external) - set(clean_external)
)

removed_internal = sorted(
    set(raw_internal) - set(clean_internal)
)

new_external = set(clean_external) - set(raw_external)
new_internal = set(clean_internal) - set(raw_internal)

assert len(removed_external) == 9
assert len(removed_internal) == 6
assert len(new_external) == 0
assert len(new_internal) == 0

changed_set = set(changed_ids)

partition_lookup = {}

for partition_name, df in {
    "anchor": anchor,
    "development": development,
    "training": clean_training,
    "pilot": pilot
}.items():
    for record_id in df["record_id"]:
        record_id = int(record_id)

        assert record_id not in partition_lookup
        partition_lookup[record_id] = partition_name

rows = []

for a, b in removed_external:
    rows.append({
        "pair_type": "EXTERNAL",
        "record_a": a,
        "record_b": b,
        "partition_b": partition_lookup[b],
        "raw_cosine": round(raw_external[(a, b)], 4),
        "a_text_changed": a in changed_set,
        "b_text_changed": b in changed_set
    })

for a, b in removed_internal:
    rows.append({
        "pair_type": "INTERNAL",
        "record_a": a,
        "record_b": b,
        "partition_b": "holdout",
        "raw_cosine": round(raw_internal[(a, b)], 4),
        "a_text_changed": a in changed_set,
        "b_text_changed": b in changed_set
    })

removed_pair_audit = pd.DataFrame(rows)

removed_pair_audit["directly_changed"] = (
    removed_pair_audit["a_text_changed"]
    | removed_pair_audit["b_text_changed"]
)

assert len(removed_pair_audit) == 15

print("S-SLTE | CANDIDATE PAIR CHANGE AUDIT")
print("-----------------------------------")

print("\nDISAPPEARED PAIRS")
display(
    removed_pair_audit.sort_values(
        ["pair_type", "record_a", "record_b"]
    ).reset_index(drop=True)
)

print("\nSUMMARY BY PAIR TYPE AND PARTITION")
display(
    removed_pair_audit.groupby(
        ["pair_type", "partition_b"]
    ).size().reset_index(name="removed_pairs")
)

print("\nCLEANING IMPACT")
print(
    "Pairs involving directly changed text:",
    int(removed_pair_audit["directly_changed"].sum())
)

print(
    "Pairs with no directly changed record:",
    int((~removed_pair_audit["directly_changed"]).sum())
)

print(
    "All 6 suspicious Solaris cluster edges removed:",
    set(
        (a, b) for a, b in removed_internal
    ).issuperset(
        {
            (95309, 106486),
            (95309, 111618),
            (95309, 112808),
            (106486, 111618),
            (106486, 112808),
            (111618, 112808)
        }
    )
)

print("\nVALIDATION STATUS")
print("Original datasets modified: NO")
print("Human decisions modified: NO")
print("Records removed: 0")
print("Reference labels accessed: NO")
print("Holdout frozen: NO")
print("Status: PAIR_DELTA_IDENTIFIED")


S-SLTE | CANDIDATE PAIR CHANGE AUDIT
-----------------------------------

DISAPPEARED PAIRS


,pair_type,record_a,record_b,partition_b,raw_cosine,a_text_changed,b_text_changed,directly_changed
0,EXTERNAL,50476,24641,training,0.8500,False,False,False
1,EXTERNAL,95309,105416,development,0.8960,True,True,True
2,EXTERNAL,95309,112806,training,0.8833,True,True,True
3,EXTERNAL,106486,105416,development,0.8832,True,True,True
4,EXTERNAL,106486,112806,training,0.8644,True,True,True
5,EXTERNAL,111618,105416,development,0.8897,True,True,True
6,EXTERNAL,111618,112806,training,0.8729,True,True,True
7,EXTERNAL,112808,105416,development,0.9064,True,True,True
8,EXTERNAL,112808,112806,training,0.8909,True,True,True
9,INTERNAL,95309,106486,holdout,0.8680,True,True,True



SUMMARY BY PAIR TYPE AND PARTITION


,pair_type,partition_b,removed_pairs
0,EXTERNAL,development,4
1,EXTERNAL,training,5
2,INTERNAL,holdout,6



CLEANING IMPACT
Pairs involving directly changed text: 14
Pairs with no directly changed record: 1
All 6 suspicious Solaris cluster edges removed: True

VALIDATION STATUS
Original datasets modified: NO
Human decisions modified: NO
Records removed: 0
Reference labels accessed: NO
Holdout frozen: NO
Status: PAIR_DELTA_IDENTIFIED


In [7]:

# S-SLTE | Step 7 — Corrected Human Review Reconciliation
# Reconstruct original Pair IDs deterministically.

import pandas as pd
import re
import html
import unicodedata
from collections import defaultdict
from difflib import SequenceMatcher

REVIEW_DIR = BASE / "outputs" / "private_human_review"

# -------------------------------------------
# 1. Load the existing human-review decisions
# -------------------------------------------

islam = pd.read_excel(
    REVIEW_DIR / "islam_review_english.xlsx"
)

yousuf = pd.read_excel(
    REVIEW_DIR / "yousuf_review_english.xlsx"
)

consensus = pd.read_excel(
    REVIEW_DIR / "consensus_adjudication_v1.xlsx"
)

required = {
    "pair_id", "article_a", "article_b",
    "decision_D_R_F_U"
}

for df in (islam, yousuf):
    assert len(df) == 71
    assert required.issubset(df.columns)
    assert df["pair_id"].is_unique

islam = islam.set_index("pair_id").sort_index()
yousuf = yousuf.set_index("pair_id").sort_index()

assert islam.index.equals(yousuf.index)

# Preserve both original reviewer decisions.
reviews = pd.DataFrame({
    "pair_id": islam.index,
    "islam_decision": (
        islam["decision_D_R_F_U"]
        .astype(str).str.strip().str.upper()
        .to_numpy()
    ),
    "yousuf_decision": (
        yousuf["decision_D_R_F_U"]
        .astype(str).str.strip().str.upper()
        .to_numpy()
    )
})

assert set(reviews["islam_decision"]) <= {"D", "R", "F", "U"}
assert set(reviews["yousuf_decision"]) <= {"D", "R", "F", "U"}

disagreement_ids = set(
    reviews.loc[
        reviews["islam_decision"]
        != reviews["yousuf_decision"],
        "pair_id"
    ]
)

assert disagreement_ids == {
    "PAIR-032", "PAIR-043", "PAIR-057"
}

assert consensus["pair_id"].is_unique
assert set(consensus["pair_id"]) == disagreement_ids
assert consensus["resolution_status"].eq("RESOLVED").all()

final_map = consensus.set_index("pair_id")[
    "final_decision_D_R_F_U"
].to_dict()

reviews["final_decision"] = reviews.apply(
    lambda r: (
        r["islam_decision"]
        if r["islam_decision"] == r["yousuf_decision"]
        else str(final_map[r["pair_id"]]).strip().upper()
    ),
    axis=1
)

assert reviews["final_decision"].value_counts().to_dict() == {
    "D": 53, "F": 17, "R": 1
}

# -------------------------------------------
# 2. Rebuild the original candidate-pair graph
# -------------------------------------------

assert len(raw_external) == 63
assert len(raw_internal) == 8

parent = {}

def find_node(node):
    if node not in parent:
        parent[node] = node
    if parent[node] != node:
        parent[node] = find_node(parent[node])
    return parent[node]

def union_nodes(a, b):
    ra, rb = find_node(a), find_node(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

for a, b in raw_external:
    union_nodes(
        ("holdout", int(a)),
        (str(partition_lookup[b]), int(b))
    )

for a, b in raw_internal:
    union_nodes(
        ("holdout", int(a)),
        ("holdout", int(b))
    )

components = defaultdict(list)

for node in list(parent):
    components[find_node(node)].append(node)

ordered_components = sorted(
    [sorted(nodes) for nodes in components.values()],
    key=lambda nodes: tuple(nodes)
)

assert len(ordered_components) == 56

node_to_component = {}
component_priority = {}

for i, nodes in enumerate(ordered_components, 1):
    cid = f"C{i:03d}"

    partitions_here = {p for p, _ in nodes}

    if partitions_here & {"anchor", "development"}:
        priority = "P1_CALIBRATION"
    elif "pilot" in partitions_here:
        priority = "P2_PILOT"
    else:
        priority = "P3_TRAINING_INTERNAL"

    component_priority[cid] = priority

    for node in nodes:
        node_to_component[node] = cid

# -------------------------------------------
# 3. Reproduce original Pair-ID ordering
# -------------------------------------------

rows = []

for (a, b), score in raw_external.items():
    cid = node_to_component[("holdout", int(a))]

    rows.append({
        "component_id": cid,
        "pair_type": "EXTERNAL",
        "record_a": int(a),
        "record_b": int(b),
        "partition_b": partition_lookup[b],
        "cosine": float(score),
        "priority": component_priority[cid]
    })

for (a, b), score in raw_internal.items():
    cid = node_to_component[("holdout", int(a))]

    rows.append({
        "component_id": cid,
        "pair_type": "INTERNAL",
        "record_a": int(a),
        "record_b": int(b),
        "partition_b": "holdout",
        "cosine": float(score),
        "priority": component_priority[cid]
    })

original_pairs = pd.DataFrame(rows)

priority_rank = {
    "P1_CALIBRATION": 0,
    "P2_PILOT": 1,
    "P3_TRAINING_INTERNAL": 2
}

original_pairs["_rank"] = (
    original_pairs["priority"].map(priority_rank)
)

original_pairs = (
    original_pairs.sort_values(
        ["_rank", "component_id", "cosine",
         "record_a", "record_b"],
        ascending=[True, True, False, True, True]
    )
    .drop(columns="_rank")
    .reset_index(drop=True)
)

original_pairs.insert(
    0,
    "pair_id",
    [f"PAIR-{i:03d}" for i in range(1, 72)]
)

assert len(original_pairs) == 71
assert original_pairs["pair_id"].is_unique

# -------------------------------------------
# 4. Validate pair-to-article correspondence
# -------------------------------------------

def normalize_article(text):
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200b", "")
    return re.sub(r"\s+", " ", text).strip().casefold()

text_lookup = all_screen.set_index("record_id")["text"]

alignment = []

for row in original_pairs.itertuples(index=False):
    pid = row.pair_id

    actual_a = normalize_article(
        text_lookup.loc[row.record_a]
    )
    actual_b = normalize_article(
        text_lookup.loc[row.record_b]
    )

    reviewed_a = normalize_article(
        islam.loc[pid, "article_a"]
    )
    reviewed_b = normalize_article(
        islam.loc[pid, "article_b"]
    )

    alignment.append({
        "pair_id": pid,
        "record_a": row.record_a,
        "record_b": row.record_b,
        "a_aligned": actual_a == reviewed_a,
        "b_aligned": actual_b == reviewed_b,
        "a_similarity": round(
            SequenceMatcher(
                None, actual_a, reviewed_a,
                autojunk=False
            ).ratio(), 4
        ),
        "b_similarity": round(
            SequenceMatcher(
                None, actual_b, reviewed_b,
                autojunk=False
            ).ratio(), 4
        )
    })

alignment_df = pd.DataFrame(alignment)

fully_aligned = (
    alignment_df["a_aligned"]
    & alignment_df["b_aligned"]
)

print("ORIGINAL PAIR-ID RECONSTRUCTION")
print("--------------------------------")
print("Candidate pairs:", len(original_pairs))
print("Review records:", len(reviews))
print("Fully text-aligned:", int(fully_aligned.sum()))

# Never silently attach decisions to unmatched pairs.
if not fully_aligned.all():

    print("\nTEXT ALIGNMENT REQUIRES INVESTIGATION")
    display(
        alignment_df.loc[
            ~fully_aligned,
            [
                "pair_id", "record_a", "record_b",
                "a_aligned", "b_aligned",
                "a_similarity", "b_similarity"
            ]
        ].head(20)
    )

    raise AssertionError(
        "Review text alignment incomplete. "
        "Do not assign decisions until resolved."
    )

# -------------------------------------------
# 5. Merge verified human review decisions
# -------------------------------------------

matched = original_pairs.merge(
    reviews[
        ["pair_id", "final_decision"]
    ],
    on="pair_id",
    validate="one_to_one"
)

removed_keys = (
    {("EXTERNAL", a, b) for a, b in removed_external}
    | {("INTERNAL", a, b) for a, b in removed_internal}
)

matched["no_longer_flagged"] = [
    (kind, a, b) in removed_keys
    for kind, a, b in zip(
        matched["pair_type"],
        matched["record_a"],
        matched["record_b"]
    )
]

affected = matched[
    matched["no_longer_flagged"]
].copy()

assert len(affected) == 15

print("\nHUMAN REVIEW RECONCILIATION PASSED")
print("Matched pair IDs:", len(matched))
print("No longer flagged:", len(affected))

print("\nAFFECTED PREVIOUS DECISIONS")
display(
    affected[
        [
            "pair_id", "pair_type",
            "record_a", "record_b",
            "partition_b", "final_decision"
        ]
    ].sort_values("pair_id").reset_index(drop=True)
)

print("\nDECISION DISTRIBUTION")
display(
    pd.crosstab(
        affected["pair_type"],
        affected["final_decision"]
    )
)

print("\nVALIDATION")
print("Original datasets modified: NO")
print("Review decisions changed: NO")
print("Reference labels accessed: NO")
print("Holdout frozen: NO")
print("Status: VERIFIED_RECONCILIATION")


ORIGINAL PAIR-ID RECONSTRUCTION
--------------------------------
Candidate pairs: 71
Review records: 71
Fully text-aligned: 71

HUMAN REVIEW RECONCILIATION PASSED
Matched pair IDs: 71
No longer flagged: 15

AFFECTED PREVIOUS DECISIONS


,pair_id,pair_type,record_a,record_b,partition_b,final_decision
0,PAIR-013,EXTERNAL,112808,105416,development,F
1,PAIR-014,EXTERNAL,95309,105416,development,F
2,PAIR-015,INTERNAL,95309,112808,holdout,F
3,PAIR-016,EXTERNAL,112808,112806,training,F
4,PAIR-017,EXTERNAL,111618,105416,development,F
5,PAIR-018,INTERNAL,111618,112808,holdout,F
6,PAIR-019,EXTERNAL,95309,112806,training,F
7,PAIR-020,EXTERNAL,106486,105416,development,F
8,PAIR-021,INTERNAL,106486,112808,holdout,F
9,PAIR-022,INTERNAL,95309,111618,holdout,F



DECISION DISTRIBUTION


final_decision,F
pair_type,
EXTERNAL,9
INTERNAL,6



VALIDATION
Original datasets modified: NO
Review decisions changed: NO
Reference labels accessed: NO
Holdout frozen: NO
Status: VERIFIED_RECONCILIATION


In [8]:

# S-SLTE | Stage 8 — Step 8

import pandas as pd
from collections import defaultdict

# 1. Keep only pairs still flagged after cleaning.
clean_keys = (
    {("EXTERNAL", int(a), int(b))
     for a, b in clean_external}
    |
    {("INTERNAL", int(a), int(b))
     for a, b in clean_internal}
)

policy_review = matched.copy()

policy_review["pair_key"] = [
    (kind, int(a), int(b))
    for kind, a, b in zip(
        policy_review["pair_type"],
        policy_review["record_a"],
        policy_review["record_b"]
    )
]

assert policy_review["pair_key"].is_unique

remaining = policy_review[
    policy_review["pair_key"].isin(clean_keys)
].copy()

assert len(remaining) == 56
assert set(remaining["pair_key"]) == clean_keys

decision_counts = (
    remaining["final_decision"]
    .value_counts().to_dict()
)

assert decision_counts == {
    "D": 53,
    "F": 2,
    "R": 1
}

# 2. Classify confirmed duplicate edges only.
duplicates = remaining[
    remaining["final_decision"] == "D"
].copy()

assert len(duplicates) == 53

protected = {"anchor", "development", "pilot"}

replacement_edges = duplicates[
    duplicates["partition_b"].isin(protected)
]

training_edges = duplicates[
    duplicates["partition_b"] == "training"
]

internal_edges = duplicates[
    duplicates["pair_type"] == "INTERNAL"
]

replacement_ids = set(
    replacement_edges["record_a"].astype(int)
)

training_exclusion_ids = set(
    training_edges["record_b"].astype(int)
)

internal_holdout_ids = (
    set(internal_edges["record_a"].astype(int))
    | set(internal_edges["record_b"].astype(int))
)

assert len(replacement_ids) == 14
assert len(training_exclusion_ids) == 37
assert len(internal_edges) == 2
assert len(internal_holdout_ids) == 4

# 3. Build components using CONFIRMED duplicate edges.
graph = defaultdict(set)

for row in duplicates.itertuples(index=False):
    a = ("holdout", int(row.record_a))
    b = (str(row.partition_b), int(row.record_b))

    graph[a].add(b)
    graph[b].add(a)

visited = set()
component_rows = []

for start in sorted(graph):
    if start in visited:
        continue

    stack = [start]
    nodes = set()

    while stack:
        node = stack.pop()

        if node in visited:
            continue

        visited.add(node)
        nodes.add(node)
        stack.extend(graph[node] - visited)

    holdout_nodes = {
        record_id for partition, record_id in nodes
        if partition == "holdout"
    }

    protected_nodes = {
        node for node in nodes
        if node[0] in protected
    }

    component_rows.append({
        "component_size": len(nodes),
        "holdout_records": len(holdout_nodes),
        "protected_records": len(protected_nodes),
        "training_records": sum(
            p == "training" for p, _ in nodes
        ),
        "has_protected": bool(protected_nodes)
    })

component_audit = pd.DataFrame(component_rows)

# 4. Check overlapping holdout policy candidates.
overlap_ids = (
    replacement_ids & internal_holdout_ids
)

assert replacement_ids.issubset(
    set(holdout["record_id"].astype(int))
)

assert internal_holdout_ids.issubset(
    set(holdout["record_id"].astype(int))
)

assert training_exclusion_ids.issubset(
    set(clean_training["record_id"].astype(int))
)

print("CONFIRMED DUPLICATE POLICY AUDIT")
print("--------------------------------")

print("\nREMAINING HUMAN DECISIONS")
display(
    remaining["final_decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="pairs")
)

print("\nDUPLICATE GRAPH SUMMARY")
print("Confirmed duplicate edges:", len(duplicates))
print("Duplicate components:", len(component_audit))

display(
    component_audit.groupby(
        ["has_protected", "holdout_records"]
    ).size().reset_index(name="components")
)

print("\nPROPOSED POLICY CANDIDATES")
display(pd.DataFrame([
    {
        "action": "HOLDOUT_REPLACEMENT_CANDIDATE",
        "unique_records": len(replacement_ids)
    },
    {
        "action": "TRAINING_EXCLUSION_CANDIDATE",
        "unique_records": len(training_exclusion_ids)
    },
    {
        "action": "INTERNAL_HOLDOUT_REVIEW",
        "unique_records": len(internal_holdout_ids)
    },
    {
        "action": "OVERLAPPING_HOLDOUT_CANDIDATES",
        "unique_records": len(overlap_ids)
    }
]))

print("\nVALIDATION STATUS")
print("Duplicate decisions changed: NO")
print("Original datasets modified: NO")
print("Records removed or replaced: 0")
print("Reference labels accessed: NO")
print("Holdout frozen: NO")
print("Policy status: DRAFT_COMPONENT_AUDIT")


CONFIRMED DUPLICATE POLICY AUDIT
--------------------------------

REMAINING HUMAN DECISIONS


,decision,pairs
0,D,53
1,F,2
2,R,1



DUPLICATE GRAPH SUMMARY
Confirmed duplicate edges: 53
Duplicate components: 51


,has_protected,holdout_records,components
0,False,1,35
1,False,2,2
2,True,1,14



PROPOSED POLICY CANDIDATES


,action,unique_records
0,HOLDOUT_REPLACEMENT_CANDIDATE,14
1,TRAINING_EXCLUSION_CANDIDATE,37
2,INTERNAL_HOLDOUT_REVIEW,4
3,OVERLAPPING_HOLDOUT_CANDIDATES,0



VALIDATION STATUS
Duplicate decisions changed: NO
Original datasets modified: NO
Records removed or replaced: 0
Reference labels accessed: NO
Holdout frozen: NO
Policy status: DRAFT_COMPONENT_AUDIT


In [9]:

# Selection is DRAFT until leakage checks pass.

import numpy as np
import pandas as pd
import hashlib
from collections import defaultdict

REPLACEMENT_SEED = 20261027

# 1. Proposed replacements caused by overlap
#    with anchor, development, or pilot.
protected_replacements = set(
    int(x) for x in replacement_ids
)

assert len(protected_replacements) == 14

# 2. Resolve internal duplicate components.
internal_dups = duplicates[
    duplicates["pair_type"] == "INTERNAL"
]

internal_graph = defaultdict(set)

for row in internal_dups.itertuples(index=False):
    a = int(row.record_a)
    b = int(row.record_b)

    internal_graph[a].add(b)
    internal_graph[b].add(a)

visited = set()
internal_groups = []

for start in sorted(internal_graph):

    if start in visited:
        continue

    stack = [start]
    group = set()

    while stack:
        node = stack.pop()

        if node in visited:
            continue

        visited.add(node)
        group.add(node)

        stack.extend(internal_graph[node] - visited)

    internal_groups.append(sorted(group))

assert len(internal_groups) == 2
assert all(len(group) == 2 for group in internal_groups)

internal_keep = {
    min(group) for group in internal_groups
}

internal_replace = {
    record_id
    for group in internal_groups
    for record_id in group
    if record_id != min(group)
}

assert len(internal_replace) == 2

# 3. Combine replacement decisions.
holdout_replace_ids = (
    protected_replacements | internal_replace
)

assert len(holdout_replace_ids) == 16

assert not (
    protected_replacements & internal_replace
)

original_holdout_ids = set(
    holdout["record_id"].astype(int)
)

original_training_ids = set(
    clean_training["record_id"].astype(int)
)

assert holdout_replace_ids.issubset(
    original_holdout_ids
)

retained_holdout_ids = (
    original_holdout_ids - holdout_replace_ids
)

assert len(retained_holdout_ids) == 1984

# 4. Exclude risky training candidates.
# Include ALL previously flagged raw external
# training matches, even those no longer flagged.
raw_training_matches = {
    int(b)
    for a, b in raw_external
    if int(b) in original_training_ids
}

excluded_training_ids = set(
    int(x) for x in training_exclusion_ids
)

assert len(excluded_training_ids) == 37

eligible_replacement_pool = (
    original_training_ids
    - excluded_training_ids
    - raw_training_matches
)

assert len(eligible_replacement_pool) >= 16

# 5. Deterministic candidate selection.
rng = np.random.default_rng(REPLACEMENT_SEED)

selected_replacement_ids = set(
    int(x)
    for x in rng.choice(
        np.array(
            sorted(eligible_replacement_pool),
            dtype=np.int64
        ),
        size=16,
        replace=False
    )
)

assert len(selected_replacement_ids) == 16

# 6. Construct DRAFT ID sets in memory.
proposed_final_holdout_ids = (
    retained_holdout_ids | selected_replacement_ids
)

proposed_final_training_ids = (
    original_training_ids
    - excluded_training_ids
    - selected_replacement_ids
)

assert len(proposed_final_holdout_ids) == 2000
assert len(proposed_final_training_ids) == 13643

assert proposed_final_holdout_ids.isdisjoint(
    proposed_final_training_ids
)

assert proposed_final_holdout_ids.isdisjoint(
    set(anchor["record_id"].astype(int))
    | set(development["record_id"].astype(int))
    | set(pilot["record_id"].astype(int))
)

def id_sha256(ids):
    payload = "\n".join(
        str(int(i)) for i in sorted(ids)
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()

# 7. Summarize without displaying article text.
print("DETERMINISTIC HOLDOUT POLICY PROPOSAL")
print("-------------------------------------")

print("\nINTERNAL DUPLICATE RESOLUTION")
for group in internal_groups:
    print(
        "Group:", group,
        "| Keep:", min(group),
        "| Replace:", max(group)
    )

print("\nPROPOSED RECORD COUNTS")
display(pd.DataFrame([
    {
        "category": "Protected-overlap replacements",
        "records": len(protected_replacements)
    },
    {
        "category": "Internal duplicate replacements",
        "records": len(internal_replace)
    },
    {
        "category": "Total holdout replacements",
        "records": len(holdout_replace_ids)
    },
    {
        "category": "Retained original holdout",
        "records": len(retained_holdout_ids)
    },
    {
        "category": "New replacement candidates",
        "records": len(selected_replacement_ids)
    },
    {
        "category": "Proposed final holdout",
        "records": len(proposed_final_holdout_ids)
    },
    {
        "category": "Proposed training",
        "records": len(proposed_final_training_ids)
    }
]))

print("\nREPLACEMENT CANDIDATE IDS")
print(sorted(selected_replacement_ids))

print("\nDRAFT FINGERPRINTS")
print(
    "Holdout SHA256:",
    id_sha256(proposed_final_holdout_ids)
)
print(
    "Training SHA256:",
    id_sha256(proposed_final_training_ids)
)

print("\nVALIDATION STATUS")
print("Deterministic seed:", REPLACEMENT_SEED)
print("Original dataset files modified: NO")
print("Reference labels accessed: NO")
print("Final leakage screening completed: NO")
print("Holdout frozen: NO")
print("Status: DRAFT_REPLACEMENT_PROPOSAL")


DETERMINISTIC HOLDOUT POLICY PROPOSAL
-------------------------------------

INTERNAL DUPLICATE RESOLUTION
Group: [56820, 56966] | Keep: 56820 | Replace: 56966
Group: [92770, 92798] | Keep: 92770 | Replace: 92798

PROPOSED RECORD COUNTS


,category,records
0,Protected-overlap replacements,14
1,Internal duplicate replacements,2
2,Total holdout replacements,16
3,Retained original holdout,1984
4,New replacement candidates,16
5,Proposed final holdout,2000
6,Proposed training,13643



REPLACEMENT CANDIDATE IDS
[1235, 4561, 16859, 26074, 27404, 42641, 45726, 50021, 52666, 71229, 75400, 86939, 87645, 110920, 115331, 118332]

DRAFT FINGERPRINTS
Holdout SHA256: 66ab8197d1396a3dca7c308cad0e2f2b13eae7a8dbda9812b2805f4cd51b4963
Training SHA256: bcb8e6000c915e6e2b163072b5d308397f318a065725a6d6e914759301bd2d9a

VALIDATION STATUS
Deterministic seed: 20261027
Original dataset files modified: NO
Reference labels accessed: NO
Final leakage screening completed: NO
Holdout frozen: NO
Status: DRAFT_REPLACEMENT_PROPOSAL


In [10]:

# S-SLTE | Stage 8 — Step 10
# Replacement Leakage Screening
# Exhaustive similarity screening for 16 new holdout records.

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Reconstruct the active proposed partitions.
active_holdout = all_screen[
    all_screen["record_id"].isin(
        proposed_final_holdout_ids
    )
].copy()

active_training = all_screen[
    all_screen["record_id"].isin(
        proposed_final_training_ids
    )
].copy()

assert len(active_holdout) == 2000
assert len(active_training) == 13643

active_parts = {
    "holdout": active_holdout,
    "training": active_training,
    "anchor": anchor,
    "development": development,
    "pilot": pilot
}

active_df = pd.concat(
    [
        df.assign(partition=name)
        for name, df in active_parts.items()
    ],
    ignore_index=True
)

assert len(active_df) == 19943
assert active_df["record_id"].is_unique

# 2. Ensure all replacement records are present.
replacement_list = sorted(selected_replacement_ids)

assert len(replacement_list) == 16
assert set(replacement_list).issubset(
    set(active_holdout["record_id"].astype(int))
)

record_ids = active_df["record_id"].astype(int).to_numpy()

id_to_index = {
    int(record_id): i
    for i, record_id in enumerate(record_ids)
}

query_indices = [
    id_to_index[record_id]
    for record_id in replacement_list
]

# 3. Screen each replacement against every other
#    active record, including other replacements.
def screen_replacement_candidates(texts, version):

    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_features=80000,
        sublinear_tf=True,
        dtype=np.float32
    )

    matrix = vectorizer.fit_transform(texts)

    # 16 queries x all active records
    similarity = (
        matrix[query_indices] @ matrix.T
    ).toarray()

    # Exclude each record's self-comparison.
    for row_idx, col_idx in enumerate(query_indices):
        similarity[row_idx, col_idx] = -1.0

    rows = []

    for row_idx, source_id in enumerate(replacement_list):

        # Review threshold is deliberately lower
        # than the original 0.85 duplicate screen.
        matches = np.flatnonzero(
            similarity[row_idx] >= 0.80
        )

        for j in matches:

            rows.append({
                "version": version,
                "replacement_id": source_id,
                "matching_id": int(record_ids[j]),
                "matching_partition": (
                    active_df.iloc[j]["partition"]
                ),
                "cosine": float(
                    similarity[row_idx, j]
                )
            })

    return pd.DataFrame(
        rows,
        columns=[
            "version", "replacement_id",
            "matching_id", "matching_partition",
            "cosine"
        ]
    )


raw_hits = screen_replacement_candidates(
    active_df["text"].astype(str).tolist(),
    "RAW"
)

cleaned_hits = screen_replacement_candidates(
    [
        candidate_clean(text)
        for text in active_df["text"]
    ],
    "CANDIDATE_CLEANED"
)

hits = pd.concat(
    [raw_hits, cleaned_hits],
    ignore_index=True
)

# 4. Exact normalized duplicate check across
#    the entire active proposed corpus.
def normalize_exact_text(value):
    import re
    import unicodedata

    return re.sub(
        r"\s+", " ",
        unicodedata.normalize(
            "NFKC", str(value)
        ).casefold()
    ).strip()

exact_texts = active_df["text"].map(
    normalize_exact_text
)

exact_collisions = active_df.loc[
    exact_texts.duplicated(keep=False),
    ["record_id", "partition"]
].copy()

# 5. Summarize the new replacement risk.
summary = []

for version, result in [
    ("RAW", raw_hits),
    ("CANDIDATE_CLEANED", cleaned_hits)
]:
    summary.append({
        "version": version,
        "pairs_cosine_ge_080": len(result),
        "pairs_cosine_ge_085": int(
            (result["cosine"] >= 0.85).sum()
        ),
        "replacement_ids_flagged": (
            result["replacement_id"].nunique()
        )
    })

summary_df = pd.DataFrame(summary)

print("S-SLTE | REPLACEMENT LEAKAGE SCREENING")
print("--------------------------------------")
print("Replacement records checked:", 16)
print("Active corpus records:", len(active_df))

print("\nSIMILARITY SCREENING SUMMARY")
display(summary_df)

print("\nFLAGGED PAIRS — METADATA ONLY")

if hits.empty:
    print("No pair reached cosine 0.80.")
else:
    display(
        hits.sort_values(
            ["version", "cosine"],
            ascending=[True, False]
        )
        .round({"cosine": 4})
        .reset_index(drop=True)
    )

print("\nEXACT-TEXT DUPLICATE CHECK")
print(
    "Active records involved in exact collisions:",
    len(exact_collisions)
)

if not exact_collisions.empty:
    display(exact_collisions.reset_index(drop=True))

print("\nVALIDATION STATUS")
print("Reference labels accessed: NO")
print("Original files modified: NO")
print("Candidate decisions automatically changed: NO")
print("Records removed: 0")
print("Holdout frozen: NO")
print("Status: REPLACEMENT_LEAKAGE_SCREENED")


S-SLTE | REPLACEMENT LEAKAGE SCREENING
--------------------------------------
Replacement records checked: 16
Active corpus records: 19943

SIMILARITY SCREENING SUMMARY


,version,pairs_cosine_ge_080,pairs_cosine_ge_085,replacement_ids_flagged
0,RAW,0,0,0
1,CANDIDATE_CLEANED,0,0,0



FLAGGED PAIRS — METADATA ONLY
No pair reached cosine 0.80.

EXACT-TEXT DUPLICATE CHECK
Active records involved in exact collisions: 0

VALIDATION STATUS
Reference labels accessed: NO
Original files modified: NO
Candidate decisions automatically changed: NO
Records removed: 0
Holdout frozen: NO
Status: REPLACEMENT_LEAKAGE_SCREENED


In [11]:

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

assert len(active_df) == 19943
assert len(active_holdout) == 2000
assert active_df["record_id"].is_unique

# active_df begins with the proposed holdout.
assert active_df.iloc[:2000]["record_id"].tolist() == (
    active_holdout["record_id"].tolist()
)

screening_texts = [
    candidate_clean(text)
    for text in active_df["text"].astype(str)
]

vectorizer_final = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_features=80000,
    sublinear_tf=True,
    dtype=np.float32
)

X_final = vectorizer_final.fit_transform(
    screening_texts
)

record_ids_final = (
    active_df["record_id"].astype(int).to_numpy()
)

partitions_final = active_df["partition"].to_numpy()

HOLDOUT_SIZE = 2000
AUDIT_THRESHOLD = 0.80
BATCH = 50

hits_by_pair = {}

for start in range(0, HOLDOUT_SIZE, BATCH):

    end = min(start + BATCH, HOLDOUT_SIZE)

    # Sparse multiplication avoids constructing
    # a full 2000 x 19943 dense matrix.
    similarities = (
        X_final[start:end] @ X_final.T
    ).tocoo()

    for i, j, score in zip(
        similarities.row,
        similarities.col,
        similarities.data
    ):
        if score < AUDIT_THRESHOLD:
            continue

        source_idx = start + int(i)
        target_idx = int(j)

        if source_idx == target_idx:
            continue

        a = int(record_ids_final[source_idx])
        b = int(record_ids_final[target_idx])

        if target_idx < HOLDOUT_SIZE:
            # Record internal pairs only once.
            if source_idx > target_idx:
                continue

            pair_type = "INTERNAL"
            pair_key = ("INTERNAL", min(a, b), max(a, b))

        else:
            pair_type = "EXTERNAL"
            pair_key = ("EXTERNAL", a, b)

        hits_by_pair[pair_key] = {
            "pair_type": pair_type,
            "record_a": pair_key[1],
            "record_b": pair_key[2],
            "partition_b": str(partitions_final[target_idx]),
            "cosine": round(float(score), 5)
        }

full_audit = pd.DataFrame(
    list(hits_by_pair.values()),
    columns=[
        "pair_type", "record_a", "record_b",
        "partition_b", "cosine"
    ]
)

# Verify all previously confirmed duplicate pairs
# are broken in the new ACTIVE partitions.
active_ids = set(record_ids_final)

old_duplicate_pairs = matched[
    matched["final_decision"] == "D"
]

assert len(old_duplicate_pairs) == 53

unresolved_old_duplicates = old_duplicate_pairs[
    old_duplicate_pairs["record_a"].isin(active_ids)
    & old_duplicate_pairs["record_b"].isin(active_ids)
]

assert unresolved_old_duplicates.empty, (
    "An old confirmed duplicate pair is "
    "still present in the active corpus."
)

# Exact duplication after candidate cleaning.
import re
import unicodedata

normalized_final = pd.Series(screening_texts).map(
    lambda text: re.sub(
        r"\s+", " ",
        unicodedata.normalize(
            "NFKC", text
        ).casefold()
    ).strip()
)

cleaned_exact_duplicates = int(
    normalized_final.duplicated(keep=False).sum()
)

print("S-SLTE | FULL HOLDOUT LEAKAGE AUDIT")
print("-----------------------------------")
print("Active holdout records:", HOLDOUT_SIZE)
print("Active corpus records:", len(active_df))
print("Similarity threshold:", AUDIT_THRESHOLD)
print("Old confirmed duplicate edges still active:",
      len(unresolved_old_duplicates))

print("\nFULL SCREENING SUMMARY")

external_count = int(
    (full_audit["pair_type"] == "EXTERNAL").sum()
) if not full_audit.empty else 0

internal_count = int(
    (full_audit["pair_type"] == "INTERNAL").sum()
) if not full_audit.empty else 0

display(pd.DataFrame([
    {
        "category": "External flagged pairs",
        "count": external_count
    },
    {
        "category": "Internal flagged pairs",
        "count": internal_count
    },
    {
        "category": "Total flagged pairs",
        "count": len(full_audit)
    },
    {
        "category": "Cleaned exact duplicate records",
        "count": cleaned_exact_duplicates
    }
]))

print("\nFLAGGED PAIRS BY PARTITION")

if full_audit.empty:
    print("No pairs reached cosine 0.80.")
else:
    display(
        full_audit.groupby(
            ["pair_type", "partition_b"]
        ).size().reset_index(name="pairs")
    )

    print("\nTOP FLAGGED PAIRS — METADATA ONLY")
    display(
        full_audit.sort_values(
            "cosine", ascending=False
        ).head(30).reset_index(drop=True)
    )

print("\nVALIDATION STATUS")
print("Original datasets modified: NO")
print("Reference labels accessed: NO")
print("All previously confirmed duplicate edges broken: YES")
print("Human decisions changed: NO")
print("Holdout frozen: NO")
print("Status: FULL_SIMILARITY_AUDIT_COMPLETED")


S-SLTE | FULL HOLDOUT LEAKAGE AUDIT
-----------------------------------
Active holdout records: 2000
Active corpus records: 19943
Similarity threshold: 0.8
Old confirmed duplicate edges still active: 0

FULL SCREENING SUMMARY


,category,count
0,External flagged pairs,19
1,Internal flagged pairs,2
2,Total flagged pairs,21
3,Cleaned exact duplicate records,0



FLAGGED PAIRS BY PARTITION


,pair_type,partition_b,pairs
0,EXTERNAL,anchor,3
1,EXTERNAL,development,2
2,EXTERNAL,pilot,2
3,EXTERNAL,training,12
4,INTERNAL,holdout,2



TOP FLAGGED PAIRS — METADATA ONLY


,pair_type,record_a,record_b,partition_b,cosine
0,EXTERNAL,7745,7610,training,0.87470
1,EXTERNAL,30128,111563,training,0.86650
2,EXTERNAL,28805,45556,training,0.86064
3,EXTERNAL,50476,24641,training,0.84998
4,EXTERNAL,102121,101969,training,0.84987
5,INTERNAL,7458,7460,holdout,0.84887
6,INTERNAL,68271,68300,holdout,0.84765
7,EXTERNAL,93272,93817,anchor,0.84582
8,EXTERNAL,106573,84150,anchor,0.83879
9,EXTERNAL,42661,42250,training,0.83848



VALIDATION STATUS
Original datasets modified: NO
Reference labels accessed: NO
All previously confirmed duplicate edges broken: YES
Human decisions changed: NO
Holdout frozen: NO
Status: FULL_SIMILARITY_AUDIT_COMPLETED


In [12]:

import pandas as pd

required_columns = {
    "pair_type", "record_a", "record_b",
    "partition_b", "cosine"
}

assert required_columns.issubset(full_audit.columns)
assert len(full_audit) == 21

keys = ["pair_type", "record_a", "record_b"]

previous_reviews = matched[
    keys + ["pair_id", "final_decision"]
].copy()

assert not previous_reviews.duplicated(keys).any()
assert not full_audit.duplicated(keys).any()

review_audit = full_audit.merge(
    previous_reviews,
    on=keys,
    how="left",
    validate="one_to_one",
    indicator=True
)

assert len(review_audit) == 21

review_audit["review_status"] = review_audit[
    "_merge"
].map({
    "both": "PREVIOUSLY_REVIEWED",
    "left_only": "NEW_REVIEW_REQUIRED",
    "right_only": "NOT_APPLICABLE"
}).astype(str)

review_audit = review_audit.drop(columns="_merge")

reviewed = review_audit[
    review_audit["review_status"] == "PREVIOUSLY_REVIEWED"
]

new_review = review_audit[
    review_audit["review_status"] == "NEW_REVIEW_REQUIRED"
]

assert len(reviewed) + len(new_review) == 21
assert reviewed["final_decision"].notna().all()
assert new_review["final_decision"].isna().all()

print("REVIEW RECONCILIATION COMPLETED")
print("Total flagged pairs:", len(review_audit))
print("Previously reviewed:", len(reviewed))
print("New review required:", len(new_review))

print("\nPREVIOUS REVIEW DECISIONS")
if reviewed.empty:
    print("None")
else:
    print(
        reviewed["final_decision"]
        .value_counts()
        .to_string()
    )

print("\nFLAGGED PAIR REGISTER")
display(
    review_audit[
        [
            "pair_type", "record_a", "record_b",
            "partition_b", "cosine",
            "pair_id", "final_decision",
            "review_status"
        ]
    ].sort_values(
        "cosine", ascending=False
    ).round({"cosine": 5}).reset_index(drop=True)
)

print("\nVALIDATION")
print("Dataset modifications: 0")
print("Reference labels accessed: NO")
print("Human decisions modified: NO")
print("Holdout frozen: NO")


REVIEW RECONCILIATION COMPLETED
Total flagged pairs: 21
Previously reviewed: 4
New review required: 17

PREVIOUS REVIEW DECISIONS
final_decision
F    3
R    1

FLAGGED PAIR REGISTER


,pair_type,record_a,record_b,partition_b,cosine,pair_id,final_decision,review_status
0,EXTERNAL,7745,7610,training,0.87470,PAIR-032,R,PREVIOUSLY_REVIEWED
1,EXTERNAL,30128,111563,training,0.86650,PAIR-043,F,PREVIOUSLY_REVIEWED
2,EXTERNAL,28805,45556,training,0.86064,PAIR-042,F,PREVIOUSLY_REVIEWED
3,EXTERNAL,50476,24641,training,0.84998,PAIR-050,F,PREVIOUSLY_REVIEWED
4,EXTERNAL,102121,101969,training,0.84987,NaN,NaN,NEW_REVIEW_REQUIRED
5,INTERNAL,7458,7460,holdout,0.84887,NaN,NaN,NEW_REVIEW_REQUIRED
6,INTERNAL,68271,68300,holdout,0.84765,NaN,NaN,NEW_REVIEW_REQUIRED
7,EXTERNAL,93272,93817,anchor,0.84582,NaN,NaN,NEW_REVIEW_REQUIRED
8,EXTERNAL,106573,84150,anchor,0.83879,NaN,NaN,NEW_REVIEW_REQUIRED
9,EXTERNAL,42661,42250,training,0.83848,NaN,NaN,NEW_REVIEW_REQUIRED



VALIDATION
Dataset modifications: 0
Reference labels accessed: NO
Human decisions modified: NO
Holdout frozen: NO


In [13]:
# Summarize conservative actions for newly flagged pairs.

assert len(new_review) == 17

protected_splits = {"anchor", "development", "pilot"}

protected_pairs = new_review[
    new_review["partition_b"].isin(protected_splits)
]

training_pairs = new_review[
    new_review["partition_b"] == "training"
]

internal_pairs = new_review[
    new_review["pair_type"] == "INTERNAL"
]

additional_holdout_replacements = set(
    protected_pairs["record_a"].astype(int)
)

additional_training_exclusions = set(
    training_pairs["record_b"].astype(int)
)

additional_internal_replacements = set(
    internal_pairs[["record_a", "record_b"]]
    .max(axis=1).astype(int)
)

all_additional_replacements = (
    additional_holdout_replacements
    | additional_internal_replacements
)

assert len(protected_pairs) == 7
assert len(training_pairs) == 8
assert len(internal_pairs) == 2
assert len(all_additional_replacements) == 9

print("CONSERVATIVE LEAKAGE POLICY — DRAFT")
print("Additional holdout replacements:",
      len(all_additional_replacements))
print("Additional training exclusions:",
      len(additional_training_exclusions))
print("Previously reviewed decisions preserved: YES")
print("Dataset files modified: NO")
print("Reference labels accessed: NO")
print("Holdout frozen: NO")
print("Next: select and screen replacement records")

CONSERVATIVE LEAKAGE POLICY — DRAFT
Additional holdout replacements: 9
Additional training exclusions: 8
Previously reviewed decisions preserved: YES
Dataset files modified: NO
Reference labels accessed: NO
Holdout frozen: NO
Next: select and screen replacement records


In [14]:

import numpy as np
import pandas as pd
import hashlib

SEED = 20261029

# Verify the previously audited partition state.
current_holdout = set(map(int, proposed_final_holdout_ids))
current_training = set(map(int, proposed_final_training_ids))

assert len(current_holdout) == 2000
assert len(current_training) == 13643
assert current_holdout.isdisjoint(current_training)
assert len(new_review) == 17

# Resolve unreviewed similarity flags conservatively.
extra_holdout_exclusions = set(
    map(int, all_additional_replacements)
)

extra_training_exclusions = set(
    map(int, additional_training_exclusions)
)

assert len(extra_holdout_exclusions) == 9
assert len(extra_training_exclusions) == 8
assert extra_holdout_exclusions <= current_holdout
assert extra_training_exclusions <= current_training

# Avoid previously flagged training records during sampling.
screened_training_matches = set(
    full_audit.loc[
        full_audit["partition_b"] == "training",
        "record_b"
    ].astype(int)
)

replacement_pool = sorted(
    current_training
    - extra_training_exclusions
    - screened_training_matches
    - raw_training_matches
    - set(map(int, flagged_ids))
)

assert len(replacement_pool) >= 9

rng = np.random.default_rng(SEED)

new_replacement_ids = set(
    map(
        int,
        rng.choice(
            np.array(replacement_pool, dtype=np.int64),
            size=9,
            replace=False
        )
    )
)

# Construct the next in-memory partition proposal.
final_holdout_candidate_ids = (
    current_holdout
    - extra_holdout_exclusions
    | new_replacement_ids
)

final_training_candidate_ids = (
    current_training
    - extra_training_exclusions
    - new_replacement_ids
)

protected_ids = (
    set(anchor["record_id"].astype(int))
    | set(development["record_id"].astype(int))
    | set(pilot["record_id"].astype(int))
)

assert len(final_holdout_candidate_ids) == 2000
assert len(final_training_candidate_ids) == 13626
assert final_holdout_candidate_ids.isdisjoint(
    final_training_candidate_ids | protected_ids
)

# Previously confirmed duplicate edges must remain separated.
candidate_active_ids = (
    final_holdout_candidate_ids
    | final_training_candidate_ids
    | protected_ids
)

confirmed_duplicates = matched[
    matched["final_decision"] == "D"
]

assert all(
    not (
        int(row.record_a) in candidate_active_ids
        and int(row.record_b) in candidate_active_ids
    )
    for row in confirmed_duplicates.itertuples()
)

assert all(
    not (
        int(row.record_a) in candidate_active_ids
        and int(row.record_b) in candidate_active_ids
    )
    for row in new_review.itertuples()
)

def partition_fingerprint(record_ids):
    content = "\n".join(
        map(str, sorted(record_ids))
    ).encode("utf-8")
    return hashlib.sha256(content).hexdigest()

summary = pd.DataFrame([
    ("Previous holdout", len(current_holdout)),
    ("Additional holdout exclusions", len(extra_holdout_exclusions)),
    ("New replacement candidates", len(new_replacement_ids)),
    ("Proposed holdout", len(final_holdout_candidate_ids)),
    ("Additional training exclusions", len(extra_training_exclusions)),
    ("Proposed training", len(final_training_candidate_ids))
], columns=["Metric", "Records"])

print("CONSERVATIVE PARTITION RESOLUTION")
display(summary)

print("\nNEW REPLACEMENT IDS")
print(sorted(new_replacement_ids))

print("\nDRAFT PARTITION FINGERPRINTS")
print("Holdout:", partition_fingerprint(final_holdout_candidate_ids))
print("Training:", partition_fingerprint(final_training_candidate_ids))

print("\nVALIDATION")
print("Prior confirmed duplicate edges separated: YES")
print("17 additional flagged edges separated: YES")
print("Partition ID overlap: NONE")
print("Reference labels accessed: NO")
print("Source datasets modified: NO")
print("Final similarity screening: PENDING")
print("Holdout frozen: NO")


CONSERVATIVE PARTITION RESOLUTION


,Metric,Records
0,Previous holdout,2000
1,Additional holdout exclusions,9
2,New replacement candidates,9
3,Proposed holdout,2000
4,Additional training exclusions,8
5,Proposed training,13626



NEW REPLACEMENT IDS
[24140, 28245, 50644, 67597, 69074, 102905, 115204, 117380, 118395]

DRAFT PARTITION FINGERPRINTS
Holdout: cbc765b7c087f35ccb563ce55bd75c74ff3b29a99c4a2792f7cd0d580d6f798d
Training: 337d11c3bd11803d51cc662324c6b39dc7494fec04872e3a9f4fb2ab1904acfe

VALIDATION
Prior confirmed duplicate edges separated: YES
17 additional flagged edges separated: YES
Partition ID overlap: NONE
Reference labels accessed: NO
Source datasets modified: NO
Final similarity screening: PENDING
Holdout frozen: NO


In [15]:

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Reconstruct the proposed active corpus.
partition_sets = {
    "holdout": final_holdout_candidate_ids,
    "training": final_training_candidate_ids,
    "anchor": set(anchor["record_id"].astype(int)),
    "development": set(development["record_id"].astype(int)),
    "pilot": set(pilot["record_id"].astype(int))
}

parts = []

for name, ids in partition_sets.items():
    subset = all_screen.loc[
        all_screen["record_id"].isin(ids),
        ["record_id", "text"]
    ].copy()

    assert len(subset) == len(ids)
    subset["partition"] = name
    parts.append(subset)

audit_corpus = pd.concat(parts, ignore_index=True)

assert len(audit_corpus) == 19926
assert audit_corpus["record_id"].is_unique
assert (audit_corpus["partition"] == "holdout").iloc[:2000].all()

record_ids = audit_corpus["record_id"].astype(int).to_numpy()
partitions = audit_corpus["partition"].to_numpy()

# Exhaustively screen holdout pairs in batches.
def audit_similarity(texts, version, threshold=0.80):
    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_features=80000,
        sublinear_tf=True,
        dtype=np.float32
    )

    matrix = vectorizer.fit_transform(texts)
    rows = []

    for start in range(0, 2000, 50):
        end = min(start + 50, 2000)
        similarities = (matrix[start:end] @ matrix.T).tocoo()

        for i, j, score in zip(
            similarities.row,
            similarities.col,
            similarities.data
        ):
            source = start + int(i)
            target = int(j)

            if score < threshold or source == target:
                continue

            if target < 2000 and source > target:
                continue

            pair_type = (
                "INTERNAL" if target < 2000 else "EXTERNAL"
            )

            a, b = int(record_ids[source]), int(record_ids[target])

            if pair_type == "INTERNAL":
                a, b = sorted((a, b))

            rows.append({
                "version": version,
                "pair_type": pair_type,
                "record_a": a,
                "record_b": b,
                "partition_b": str(partitions[target]),
                "cosine": float(score)
            })

    return pd.DataFrame(
        rows,
        columns=[
            "version", "pair_type", "record_a",
            "record_b", "partition_b", "cosine"
        ]
    )

raw_audit = audit_similarity(
    audit_corpus["text"].astype(str).tolist(),
    "RAW"
)

cleaned_audit = audit_similarity(
    [
        candidate_clean(text)
        for text in audit_corpus["text"].astype(str)
    ],
    "CANDIDATE_CLEANED"
)

final_audit_draft = pd.concat(
    [raw_audit, cleaned_audit],
    ignore_index=True
)

# Reconcile flagged pairs with existing human decisions.
review_register = matched[
    [
        "pair_type", "record_a", "record_b",
        "pair_id", "final_decision"
    ]
].drop_duplicates(
    ["pair_type", "record_a", "record_b"]
)

final_audit_draft = final_audit_draft.merge(
    review_register,
    on=["pair_type", "record_a", "record_b"],
    how="left",
    validate="many_to_one"
)

final_audit_draft["review_status"] = np.where(
    final_audit_draft["final_decision"].notna(),
    "PREVIOUSLY_REVIEWED",
    "UNREVIEWED_SIMILARITY_FLAG"
)

# Compare raw and cleaned screening outcomes.
summary = (
    final_audit_draft.groupby(
        ["version", "review_status"]
    ).size().unstack(fill_value=0)
)

print("FINAL HOLDOUT SCREENING — DRAFT")
print("Active records:", len(audit_corpus))
print("Holdout records:", len(partition_sets["holdout"]))
print("Training records:", len(partition_sets["training"]))

print("\nSCREENING SUMMARY")
display(summary)

print("\nFLAGGED PAIRS BY PARTITION")
display(
    final_audit_draft.groupby(
        ["version", "pair_type", "partition_b"]
    ).size().reset_index(name="pairs")
)

print("\nNEW REPLACEMENT RECORD FLAGS")
replacement_flags = final_audit_draft[
    final_audit_draft["record_a"].isin(new_replacement_ids)
    | final_audit_draft["record_b"].isin(new_replacement_ids)
]

print("Flagged pairs involving new replacements:",
      len(replacement_flags))

if not replacement_flags.empty:
    display(
        replacement_flags.sort_values(
            "cosine", ascending=False
        ).head(25).round({"cosine": 5})
    )

print("\nUNREVIEWED CLEANED-TEXT FLAGS")
unreviewed_cleaned = final_audit_draft[
    (final_audit_draft["version"] == "CANDIDATE_CLEANED")
    & (final_audit_draft["review_status"] ==
       "UNREVIEWED_SIMILARITY_FLAG")
]

print("Remaining candidate pairs:", len(unreviewed_cleaned))

if not unreviewed_cleaned.empty:
    display(
        unreviewed_cleaned.sort_values(
            "cosine", ascending=False
        ).round({"cosine": 5})
    )

print("\nSTATUS: SCREENING COMPLETE — HOLDOUT NOT FROZEN")


FINAL HOLDOUT SCREENING — DRAFT
Active records: 19926
Holdout records: 2000
Training records: 13626

SCREENING SUMMARY


review_status,PREVIOUSLY_REVIEWED,UNREVIEWED_SIMILARITY_FLAG
version,,
CANDIDATE_CLEANED,3,5
RAW,17,0



FLAGGED PAIRS BY PARTITION


,version,pair_type,partition_b,pairs
0,CANDIDATE_CLEANED,EXTERNAL,anchor,1
1,CANDIDATE_CLEANED,EXTERNAL,training,7
2,RAW,EXTERNAL,development,4
3,RAW,EXTERNAL,training,7
4,RAW,INTERNAL,holdout,6



NEW REPLACEMENT RECORD FLAGS
Flagged pairs involving new replacements: 0

UNREVIEWED CLEANED-TEXT FLAGS
Remaining candidate pairs: 5


,version,pair_type,record_a,record_b,partition_b,cosine,pair_id,final_decision,review_status
21,CANDIDATE_CLEANED,EXTERNAL,31935,32075,training,0.82767,NaN,NaN,UNREVIEWED_SIMILARITY_FLAG
22,CANDIDATE_CLEANED,EXTERNAL,47132,47109,training,0.82346,NaN,NaN,UNREVIEWED_SIMILARITY_FLAG
24,CANDIDATE_CLEANED,EXTERNAL,55190,53788,training,0.81392,NaN,NaN,UNREVIEWED_SIMILARITY_FLAG
18,CANDIDATE_CLEANED,EXTERNAL,8102,8165,training,0.81082,NaN,NaN,UNREVIEWED_SIMILARITY_FLAG
19,CANDIDATE_CLEANED,EXTERNAL,22201,22107,anchor,0.80507,NaN,NaN,UNREVIEWED_SIMILARITY_FLAG



STATUS: SCREENING COMPLETE — HOLDOUT NOT FROZEN


In [16]:

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Apply the conservative policy to unreviewed cleaned-text flags.
assert len(unreviewed_cleaned) == 5
assert set(unreviewed_cleaned["pair_type"]) == {"EXTERNAL"}

remaining_holdout_exclusions = set(
    unreviewed_cleaned.loc[
        unreviewed_cleaned["partition_b"] == "anchor",
        "record_a"
    ].astype(int)
)

remaining_training_exclusions = set(
    unreviewed_cleaned.loc[
        unreviewed_cleaned["partition_b"] == "training",
        "record_b"
    ].astype(int)
)

assert len(remaining_holdout_exclusions) == 1
assert len(remaining_training_exclusions) == 4

# Screen replacement candidates against the current active corpus.
texts = [
    candidate_clean(text)
    for text in audit_corpus["text"].astype(str)
]

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_features=80000,
    sublinear_tf=True,
    dtype=np.float32
)

matrix = vectorizer.fit_transform(texts)

lookup = {
    int(record_id): index
    for index, record_id in enumerate(
        audit_corpus["record_id"]
    )
}

eligible_ids = sorted(
    final_training_candidate_ids
    - remaining_training_exclusions
    - raw_training_matches
    - screened_training_matches
    - set(map(int, flagged_ids))
)

rng = np.random.default_rng(20261029)
candidate_order = rng.permutation(eligible_ids)

safe_replacement_id = None

for record_id in candidate_order:
    index = lookup[int(record_id)]
    similarity = (
        matrix[index] @ matrix.T
    ).toarray().ravel()

    similarity[index] = -1.0

    if similarity.max() < 0.80:
        safe_replacement_id = int(record_id)
        break

assert safe_replacement_id is not None

# Update only in-memory partition candidates.
revised_holdout_ids = (
    final_holdout_candidate_ids
    - remaining_holdout_exclusions
    | {safe_replacement_id}
)

revised_training_ids = (
    final_training_candidate_ids
    - remaining_training_exclusions
    - {safe_replacement_id}
)

assert len(revised_holdout_ids) == 2000
assert len(revised_training_ids) == 13621
assert revised_holdout_ids.isdisjoint(revised_training_ids)
assert revised_holdout_ids.isdisjoint(protected_ids)

print("FINAL CONSERVATIVE RESOLUTION — DRAFT")
print("Holdout exclusions:", sorted(remaining_holdout_exclusions))
print("Training exclusions:", sorted(remaining_training_exclusions))
print("Selected replacement:", safe_replacement_id)
print("Proposed holdout:", len(revised_holdout_ids))
print("Proposed training:", len(revised_training_ids))
print("Holdout SHA256:", partition_fingerprint(revised_holdout_ids))
print("Training SHA256:", partition_fingerprint(revised_training_ids))
print("Candidate similarity threshold: PASSED")
print("Source datasets modified: NO")
print("Reference labels accessed: NO")
print("Final re-screening: PENDING")
print("Holdout frozen: NO")


FINAL CONSERVATIVE RESOLUTION — DRAFT
Holdout exclusions: [22201]
Training exclusions: [8165, 32075, 47109, 53788]
Selected replacement: 30984
Proposed holdout: 2000
Proposed training: 13621
Holdout SHA256: 0c529eaddf7abfd977dcad8a1627bab1369460d5846370fe64304600506609c4
Training SHA256: ce13626940296ed9158f4dc116580987f404114f66d42fdc6dcc87acf7c62625
Candidate similarity threshold: PASSED
Source datasets modified: NO
Reference labels accessed: NO
Final re-screening: PENDING
Holdout frozen: NO


In [17]:

import numpy as np
import pandas as pd
import unicodedata
import re
from sklearn.feature_extraction.text import TfidfVectorizer

# Verify the proposed partition manifest.
assert len(revised_holdout_ids) == 2000
assert len(revised_training_ids) == 13621

assert partition_fingerprint(revised_holdout_ids) == (
    "0c529eaddf7abfd977dcad8a1627bab1369460d5846370fe64304600506609c4"
)

assert partition_fingerprint(revised_training_ids) == (
    "ce13626940296ed9158f4dc116580987f404114f66d42fdc6dcc87acf7c62625"
)

partition_ids = {
    "holdout": revised_holdout_ids,
    "training": revised_training_ids,
    "anchor": set(anchor["record_id"].astype(int)),
    "development": set(development["record_id"].astype(int)),
    "pilot": set(pilot["record_id"].astype(int))
}

# Reconstruct the active corpus using text-only records.
frames = []

for name, ids in partition_ids.items():
    frame = all_screen.loc[
        all_screen["record_id"].isin(ids),
        ["record_id", "text"]
    ].copy()

    assert len(frame) == len(ids)
    frame["partition"] = name
    frames.append(frame)

validated_corpus = pd.concat(frames, ignore_index=True)

assert len(validated_corpus) == 19921
assert validated_corpus["record_id"].is_unique

ids = validated_corpus["record_id"].astype(int).to_numpy()
partitions_array = validated_corpus["partition"].to_numpy()

# Screen all holdout records against the active corpus.
def screen_holdout(texts, version, threshold=0.80):
    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_features=80000,
        sublinear_tf=True,
        dtype=np.float32
    )

    matrix = vectorizer.fit_transform(texts)
    results = []

    for start in range(0, 2000, 50):
        similarities = (
            matrix[start:start + 50] @ matrix.T
        ).tocoo()

        for i, j, value in zip(
            similarities.row,
            similarities.col,
            similarities.data
        ):
            source = start + int(i)
            target = int(j)

            if source == target or value < threshold:
                continue

            if target < 2000 and source > target:
                continue

            kind = "INTERNAL" if target < 2000 else "EXTERNAL"
            a, b = int(ids[source]), int(ids[target])

            if kind == "INTERNAL":
                a, b = sorted((a, b))

            results.append({
                "version": version,
                "pair_type": kind,
                "record_a": a,
                "record_b": b,
                "partition_b": str(partitions_array[target]),
                "cosine": float(value)
            })

    return pd.DataFrame(results, columns=[
        "version", "pair_type", "record_a",
        "record_b", "partition_b", "cosine"
    ])


raw_texts = validated_corpus["text"].astype(str).tolist()
cleaned_texts = [candidate_clean(t) for t in raw_texts]

raw_result = screen_holdout(raw_texts, "RAW")
clean_result = screen_holdout(cleaned_texts, "CANDIDATE_CLEANED")

final_screen = pd.concat(
    [raw_result, clean_result],
    ignore_index=True
)

review_columns = [
    "pair_type", "record_a", "record_b",
    "final_decision"
]

final_screen = final_screen.merge(
    matched[review_columns].drop_duplicates(
        ["pair_type", "record_a", "record_b"]
    ),
    on=["pair_type", "record_a", "record_b"],
    how="left",
    validate="many_to_one"
)

# Audit normalized exact-text collisions across active records.
def normalized_text(value):
    return re.sub(
        r"\s+", " ",
        unicodedata.normalize(
            "NFKC", str(value)
        ).casefold()
    ).strip()

exact_counts = {
    "RAW": int(
        pd.Series(raw_texts).map(normalized_text)
        .duplicated(keep=False).sum()
    ),
    "CANDIDATE_CLEANED": int(
        pd.Series(cleaned_texts).map(normalized_text)
        .duplicated(keep=False).sum()
    )
}

active_ids = set(ids)

confirmed_edges_remaining = sum(
    int(row.record_a) in active_ids
    and int(row.record_b) in active_ids
    for row in matched[
        matched["final_decision"] == "D"
    ].itertuples()
)

assert confirmed_edges_remaining == 0

summary = pd.DataFrame([
    {
        "version": version,
        "flagged_pairs": len(group),
        "unreviewed_pairs": int(
            group["final_decision"].isna().sum()
        ),
        "confirmed_duplicate_pairs": int(
            group["final_decision"].eq("D").sum()
        ),
        "exact_duplicate_records": exact_counts[version]
    }
    for version, group in [
        ("RAW", raw_result.merge(
            matched[review_columns],
            on=["pair_type", "record_a", "record_b"],
            how="left"
        )),
        ("CANDIDATE_CLEANED", clean_result.merge(
            matched[review_columns],
            on=["pair_type", "record_a", "record_b"],
            how="left"
        ))
    ]
])

print("FINAL PARTITION VALIDATION")
print("Active records:", len(validated_corpus))
print("Holdout:", len(revised_holdout_ids))
print("Training:", len(revised_training_ids))
print("Confirmed duplicate edges remaining:",
      confirmed_edges_remaining)

print("\nSIMILARITY AND EXACT-DUPLICATE AUDIT")
display(summary)

print("\nREMAINING FLAGGED PAIRS")
if final_screen.empty:
    print("None at cosine >= 0.80")
else:
    display(
        final_screen.sort_values(
            ["version", "cosine"],
            ascending=[True, False]
        ).round({"cosine": 5}).reset_index(drop=True)
    )

print("\nVALIDATION STATUS")
print("Partition ID overlap: NONE")
print("Reference labels accessed: NO")
print("Source datasets modified: NO")
print("Final partition manifest saved: NO")
print("Holdout frozen: NO")


FINAL PARTITION VALIDATION
Active records: 19921
Holdout: 2000
Training: 13621
Confirmed duplicate edges remaining: 0

SIMILARITY AND EXACT-DUPLICATE AUDIT


,version,flagged_pairs,unreviewed_pairs,confirmed_duplicate_pairs,exact_duplicate_records
0,RAW,17,0,0,0
1,CANDIDATE_CLEANED,3,0,0,0



REMAINING FLAGGED PAIRS


,version,pair_type,record_a,record_b,partition_b,cosine,final_decision
0,CANDIDATE_CLEANED,EXTERNAL,7745,7610,training,0.87470,R
1,CANDIDATE_CLEANED,EXTERNAL,30128,111563,training,0.86652,F
2,CANDIDATE_CLEANED,EXTERNAL,50476,24641,training,0.85011,F
3,RAW,EXTERNAL,112808,105416,development,0.90632,F
4,RAW,EXTERNAL,95309,105416,development,0.89599,F
5,RAW,INTERNAL,95309,112808,holdout,0.89194,F
6,RAW,EXTERNAL,111618,105416,development,0.88974,F
7,RAW,EXTERNAL,112808,112806,training,0.88848,F
8,RAW,INTERNAL,111618,112808,holdout,0.88612,F
9,RAW,EXTERNAL,106486,105416,development,0.88321,F



VALIDATION STATUS
Partition ID overlap: NONE
Reference labels accessed: NO
Source datasets modified: NO
Final partition manifest saved: NO
Holdout frozen: NO


In [18]:

import json
import hashlib
from datetime import date

# Verify the partition state before freezing IDs.
assert len(validated_corpus) == 19921
assert len(revised_holdout_ids) == 2000
assert len(revised_training_ids) == 13621
assert confirmed_edges_remaining == 0

assert exact_counts == {
    "RAW": 0,
    "CANDIDATE_CLEANED": 0
}

assert len(raw_result) == 17
assert len(clean_result) == 3
assert final_screen["final_decision"].isin(["F", "R"]).all()

partition_map = {
    "holdout": set(map(int, revised_holdout_ids)),
    "training": set(map(int, revised_training_ids)),
    "anchor": set(anchor["record_id"].astype(int)),
    "development": set(development["record_id"].astype(int)),
    "pilot": set(pilot["record_id"].astype(int))
}

all_partition_ids = set().union(*partition_map.values())

assert len(all_partition_ids) == sum(
    len(ids) for ids in partition_map.values()
)
assert len(all_partition_ids) == 19921

# Preserve source-text integrity without exporting article content.
def raw_text_fingerprint(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values("record_id").itertuples():
        digest.update(str(int(row.record_id)).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(row.text).encode("utf-8"))
        digest.update(b"\0")

    return digest.hexdigest()

frozen_partitions = {}

for name, ids in partition_map.items():
    frame = validated_corpus.loc[
        validated_corpus["partition"] == name,
        ["record_id", "text"]
    ]

    assert set(frame["record_id"].astype(int)) == ids

    frozen_partitions[name] = {
        "count": len(ids),
        "record_ids": sorted(ids),
        "id_sha256": partition_fingerprint(ids),
        "raw_text_sha256": raw_text_fingerprint(frame)
    }

original_holdout = set(map(int, original_holdout_ids))
original_training = set(map(int, original_training_ids))

removed_holdout = original_holdout - partition_map["holdout"]
added_holdout = partition_map["holdout"] - original_holdout
excluded_training = (
    original_training
    - partition_map["training"]
    - added_holdout
)

assert len(removed_holdout) == 26
assert len(added_holdout) == 26
assert len(excluded_training) == 49

manifest = {
    "schema_version": "1.0",
    "manifest_version": "v1",
    "created_date": "2026-10-29",
    "dataset": "AG News",
    "research_method": "S-SLTE",
    "status": "PARTITION_IDS_FROZEN_PENDING_SUPERVISOR_REVIEW",
    "source_text_policy": "ORIGINAL_TEXT_UNMODIFIED",
    "screening_text_policy": "CANDIDATE_CLEANING_NOT_FINALIZED",
    "holdout_selection_seed": 20261025,
    "replacement_seeds": [20261027, 20261029],
    "partitions": frozen_partitions,
    "resolution": {
        "original_holdout_replacements": len(removed_holdout),
        "training_to_holdout_transfers": len(added_holdout),
        "training_exclusions": len(excluded_training),
        "removed_holdout_ids": sorted(removed_holdout),
        "added_holdout_ids": sorted(added_holdout),
        "excluded_training_ids": sorted(excluded_training)
    },
    "final_screening": {
        "threshold": 0.80,
        "raw_flagged_pairs": len(raw_result),
        "candidate_cleaned_flagged_pairs": len(clean_result),
        "unreviewed_pairs": 0,
        "confirmed_duplicate_edges_remaining": 0,
        "raw_exact_duplicate_records": 0,
        "cleaned_exact_duplicate_records": 0
    },
    "limitations": [
        "TF-IDF screening cannot exclude every semantic duplicate.",
        "Additional similarity flags were conservatively excluded without manual adjudication.",
        "Candidate text-cleaning policy is not finalized.",
        "Supervisor methodological review remains pending."
    ]
}

# Create a private, versioned manifest without overwriting it.
manifest_dir = BASE / "configs"
manifest_dir.mkdir(parents=True, exist_ok=True)

manifest_path = manifest_dir / "holdout_partition_manifest_v1.json"

if manifest_path.exists():
    with manifest_path.open("r", encoding="utf-8") as file:
        existing_manifest = json.load(file)

    assert existing_manifest == manifest, (
        "Existing v1 manifest differs. "
        "Do not overwrite a frozen research artifact."
    )
    save_status = "EXISTING_MANIFEST_VERIFIED"
else:
    with manifest_path.open("x", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2, ensure_ascii=False)
    save_status = "NEW_MANIFEST_SAVED"

with manifest_path.open("r", encoding="utf-8") as file:
    verified_manifest = json.load(file)

assert verified_manifest == manifest

print("S-SLTE PARTITION MANIFEST")
print("Status:", save_status)
print("Version:", manifest["manifest_version"])
print("Saved to:", manifest_path)

print("\nFROZEN PARTITION COUNTS")
for name, details in frozen_partitions.items():
    print(f"{name}: {details['count']}")

print("\nFINAL ID FINGERPRINTS")
print("Holdout:", frozen_partitions["holdout"]["id_sha256"])
print("Training:", frozen_partitions["training"]["id_sha256"])

print("\nINTEGRITY")
print("Partition overlap: NONE")
print("Unresolved screening flags: 0")
print("Source CSV files modified: NO")
print("Reference labels accessed: NO")
print("Partition ID manifest: FROZEN V1")
print("Supervisor review: PENDING")


S-SLTE PARTITION MANIFEST
Status: NEW_MANIFEST_SAVED
Version: v1
Saved to: /content/drive/MyDrive/S_SLTE_Thesis/configs/holdout_partition_manifest_v1.json

FROZEN PARTITION COUNTS
holdout: 2000
training: 13621
anchor: 2000
development: 1800
pilot: 500

FINAL ID FINGERPRINTS
Holdout: 0c529eaddf7abfd977dcad8a1627bab1369460d5846370fe64304600506609c4
Training: ce13626940296ed9158f4dc116580987f404114f66d42fdc6dcc87acf7c62625

INTEGRITY
Partition overlap: NONE
Unresolved screening flags: 0
Source CSV files modified: NO
Reference labels accessed: NO
Partition ID manifest: FROZEN V1
Supervisor review: PENDING
